# Hello, quantum 👋

The same experiment in three frameworks: a **Bell state**, the simplest example of entanglement.

1. Put qubit 0 in superposition with a Hadamard gate (**H**).
2. Entangle qubit 1 with it using a controlled-NOT (**CNOT**).
3. Measure both.

You should get `00` about half the time and `11` the other half, and **never** `01` or `10`. The two qubits always agree.

Run each cell with **Shift + Enter**.

In [ ]:
import warnings
# pyQuil 4.22+ prints deprecation notices about pyQuil 5; they don't affect this lab.
warnings.filterwarnings("ignore", message=".*deprecated.*")

SHOTS = 1000

## 1 · Qiskit (IBM)

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()
qc.draw("mpl")

In [ ]:
sim = AerSimulator()
qiskit_counts = sim.run(transpile(qc, sim), shots=SHOTS).result().get_counts()
print(qiskit_counts)
plot_histogram(qiskit_counts)

## 2 · Cirq (Google)

In [ ]:
import cirq

q0, q1 = cirq.LineQubit.range(2)
circuit = cirq.Circuit(
    cirq.H(q0),
    cirq.CNOT(q0, q1),
    cirq.measure(q0, q1, key="m"),
)
print(circuit)

In [ ]:
result = cirq.Simulator().run(circuit, repetitions=SHOTS)
cirq_counts = {format(k, "02b"): v for k, v in result.histogram(key="m").items()}
print(cirq_counts)
plot_histogram(cirq_counts)

## 3 · pyQuil (Rigetti)

With `docker compose up`, this uses Rigetti's **QVM** simulator and **quilc** compiler running in their own containers. Anywhere else (for example the `uv` one-liner), it falls back to pyQuil's built-in Python simulator.

In [ ]:
import collections
from pyquil import Program, get_qc
from pyquil.gates import H, CNOT, MEASURE

program = Program(H(0), CNOT(0, 1))
ro = program.declare("ro", "BIT", 2)
program += MEASURE(0, ro[0])
program += MEASURE(1, ro[1])
program.wrap_in_numshots_loop(SHOTS)
print(program)

In [ ]:
import threading

def run_with_time_limit(fn, seconds):
    """Run fn() but give up after `seconds`. pyQuil's quilc check can wait forever."""
    out = {}
    def target():
        try:
            out["value"] = fn()
        except BaseException as exc:
            out["error"] = exc
    worker = threading.Thread(target=target, daemon=True)
    worker.start()
    worker.join(seconds)
    if worker.is_alive():
        raise TimeoutError(f"no answer from QVM/quilc within {seconds}s")
    if "error" in out:
        raise out["error"]
    return out["value"]

def run_on_servers():
    qc = get_qc("2q-qvm", compiler_timeout=20, execution_timeout=20)
    executable = qc.compile(program)          # compiled by quilc
    return qc.run(executable).get_register_map()["ro"]

try:
    bits = run_with_time_limit(run_on_servers, seconds=30)
    backend = "QVM + quilc (Docker)"
except Exception as exc:
    print(f"QVM/quilc not available ({type(exc).__name__}: {exc})")
    print("Falling back to pyQuil's built-in simulator.\n")
    bits = get_qc("2q-pyqvm").run(program).get_register_map()["ro"]
    backend = "PyQVM (built-in)"

pyquil_counts = dict(collections.Counter("".join(str(int(b)) for b in row) for row in bits))
print(f"Backend: {backend}\n{pyquil_counts}")
plot_histogram(pyquil_counts)

## Side by side

In [ ]:
plot_histogram(
    [qiskit_counts, cirq_counts, pyquil_counts],
    legend=["Qiskit", "Cirq", "pyQuil"],
    title="Bell state, three frameworks",
)

## Bonus · Grover's search in Qiskit

Grover's algorithm finds a marked item faster than checking items one by one. With 2 qubits there are 4 possible items, and a single Grover iteration finds the marked one (`11` here) with 100% probability.

In [ ]:
grover = QuantumCircuit(2)
grover.h([0, 1])          # equal superposition of 00, 01, 10, 11

grover.cz(0, 1)           # oracle: flip the phase of |11>

grover.h([0, 1])          # diffuser: reflect about the average
grover.x([0, 1])
grover.cz(0, 1)
grover.x([0, 1])
grover.h([0, 1])

grover.measure_all()
counts = sim.run(transpile(grover, sim), shots=SHOTS).result().get_counts()
print(counts)
grover.draw("mpl")

## Where next

- Add your own notebooks to this folder. They're saved on your computer, not inside the container.
- [IBM Quantum Learning](https://learning.quantum.ibm.com/) for Qiskit courses
- [Cirq tutorials](https://quantumai.google/cirq/start/basics)
- [pyQuil docs](https://pyquil-docs.rigetti.com/)